# Part 2 Implementing and Optimizing LeNet-5 with Vitis HLS

## 2.1 Vitis HLS implementation Overview

This implementation uses int2 quantization to extremely minimize resource usage and increase data parallelism. The design incorporates several advanced Vitis HLS features and optimization techniques.  

### Interface Definition

The top-level function `lenet5` has the following interface ports:

- **Input Ports:**
    - `in[HEIGHT * WIDTH]`: Input image array
    - `conv0_weight[FILTER * KERNEL * KERNEL]`: Weights for first convolution layer
    - `conv1_weight[FILTER * KERNEL * KERNEL * CHANNEL]`: Weights for second convolution layer
    - `matmul0_weight[FLATTEN * CLASS]`: Weights for fully connected layer

- **Output Port:**
    - `out`: Output stream of type `hls::stream<int32_pkt>`

All ports are configured using AXI interfaces:
- Control signals use AXI-Lite interface (s_axilite)
- Data ports use AXI-Stream interface (axis)

```cpp
void lenet5(
	int in[HEIGHT * WIDTH],                                     // input image
	int conv0_weight[FILTER * KERNEL * KERNEL],                 // conv0 weights
	int conv1_weight[FILTER * KERNEL * KERNEL * CHANNEL],       // conv1 weights
	int matmul0_weight[FLATTEN * CLASS],                        // fc weights
	hls::stream<int32_pkt>& out)                                // output stream
{
#pragma HLS INTERFACE mode=s_axilite bundle=CTRL port=return
#pragma HLS interface axis port=in
#pragma HLS interface axis port=conv0_weight
#pragma HLS interface axis port=conv1_weight
#pragma HLS interface axis port=matmul0_weight
#pragma HLS interface axis port=out
    //...existing code...
}

```
### Architecture Design

We can use the Dataflow Viewer of Vitis Unified IDE to learn about the dataflow of Vitis HLS design directly. The dataflow graph of this implementation is as follow:

![vitis_dataflow_graph](image/vitis_dataflow.png)

The maxpool layer is split into two parallel tasks: `compute_h()` and `compute_v()`, and template functions `read_input`, `pass_through`, and `write_result` are used to handle data processing.

- `read_input` Function: Takes input image data and converts it into sliding windows for convolution, uses LineBuffer template to efficiently implement the sliding window operation.

- `pass_through` Function: Serves as an adapter between convolution layers, transforms feature maps into sliding windows for next convolution.

- `write_result` Function: Accumulates partial sums from dense layer computations, formats final results into AXI-Stream packets.

```cpp
    //...existing code...
#pragma HLS dataflow
	read_input<28, 28, 5, 5>(in, ins);
	conv0.compute<int2x16_t>(ins, pips1);
	maxpool0.compute_h(pips1, pips2);
	maxpool0.compute_v(pips2, pips3);
	pass_through<12, 12, 16, 5, 5>(pips3, pips4);
	conv1.compute<int2x16_t>(pips4, pips5);
	maxpool1.compute_h(pips5, pips6);
	maxpool1.compute_v(pips6, pips7);
	matmul0.compute_muladd(pips7, outs);
	write_result<256, 10, 16>(out, outs);
```



## 2.2 The use of C++ templates

Vitis HLS supports the use of templates in C++ for synthesis. Templates allow writing generic code that works with different data types and sizes. In this implementation, the bit manipulation functions, line buffers, and convolution layers are all templated to handle different configurations. 

In Vitis HLS, template functions with different template parameter values create different instances with their own static variables. Vitis HLS synthesizes each instance independently according to its context, which is beneficial for optimizing each instance separately. You can refer to [UG1399: Vitis High-Level Synthesis User Guide](https://docs.amd.com/r/en-US/ug1399-vitis-hls/Using-Templates-to-Create-Unique-Instances) for more details.
 
Here's the example usage of template in the code:

1. The `get/getu` and `set/setu` functions:
    - Template parameter `S` defines the size of the source array(each element is `int2_t` or `uint2_t`)
    - Used for bit-level access and manipulation of ap_uint data types
    - Uses bit selection operator `src(p+2-1, p)` provided by Vitis HLS to select bits from position p to p + 2 - 1
    - Enables efficient handling of packed 2-bit values

2. The `multiply_add` function:
    - Template parameter `S` defines the size of the source array `vu` and `wi`

3. The `LineBuffer` class:
    - Template parameters define kernel height (KH), width (W), data type (T), and window type (WT)
    - Implements sliding window operation for 2D convolution, see previous `conv_filter` PBL_HLS tutorial for more details
    - Use `hls::vector<T, N>` to create a Vitis HLS vector of `N` elements of type `T`

4. The `Conv2D` class:
    - Templates parametrize height (H), width (W), channels (C), kernel height (KH), kernel width (KW), number of filters (F) and number of thresholds (M)
    - Enables reuse of convolution logic across different layer configurations
    - Supports different optimization strategies through inheritance

5. The `MaxPool2x2` class:
    - Templates configure height (H), width (W), and channels (C) for processing 2D feature maps:
        - Data type T for input/output values
        - Height and width dimensions of input feature map
        - Number of channels to process in parallel
    - Allows reuse of maxpooling logic across different layer sizes

6. The `Dense` class:
    - Templates configure input data type (IT), output data type (OT), feature length (FL), class length (CL) and packing factor (K)

7. The `read_input`, `pass_through` and `write_result` template functions:
    - These functions use template to create unique instances for different template parameters, and partition the code into a Load-Compute-Store pattern
    - Template parameters customize channel width, dimensions, and data types for each function instance:
        - `read_input<H,W,KH,KW>`: Configures window dimensions for input layer
        - `pass_through<H,W,C,KH,KW>`: Adapts dimensions between convolution layers
        - `write_result<N,M,C>`: Sets output dimension and data format

Using template classes and functions, we can easily extend the current LeNet-5 implementation to more complex convolutional neural networks. For example, we can inherit from the Conv2D class to create new convolution layers that support different data dimensions.

## 2.3 Task-Level Parallelism

Task-level parallelism is a concept that we've covered in previous tutorials. Let's review the key points:

1. Task-level parallelism allows multiple independent tasks to execute concurrently in hardware
2. In Vitis HLS, this is achieved through:
    - Create a streaming dataflow architecture(`#pragma HLS dataflow`), which allows functions to run concurrently in a producer-consumer fashion.
    - Functions are compiled into concurrent hardware blocks.
    - Use channels(typically FIFOs) for inter-task communication.


This approach is particularly valuable in streaming applications like our LeNet-5 implementation, where data flows through multiple processing stages. The LeNet-5 implementation demonstrates excellent task-level parallelism through its dataflow architecture. Here's how it's structured:

1. FIFO channels are defined to connect processing stages:
```cpp
fifo<Window_0> ins("input_fifo");
fifo<int2x16_t> pips1("pipe_fifo1");
fifo<int2x16_t> pips2("pipe_fifo2");
fifo<int2x16_t> pips3("pipe_fifo3");
fifo<Window_1> pips4("pipe_fifo4");
fifo<int2x16_t> pips5("pipe_fifo5");
fifo<int2x16_t> pips6("pipe_fifo6");
fifo<int2x16_t> pips7("pipe_fifo7");
fifo<int16_t> outs("output_fifo");
```

2. The dataflow region is established with the pragma:
```cpp
#pragma HLS dataflow
```

3. Tasks are connected in a streaming pipeline, and communicate with other tasks through FIFO channels: 
```cpp
read_input<28, 28, 5, 5>(in, ins);
conv0.compute<int2x16_t>(ins, pips1);
maxpool0.compute_h(pips1, pips2);
maxpool0.compute_v(pips2, pips3);
pass_through<12, 12, 16, 5, 5>(pips3, pips4);
conv1.compute<int2x16_t>(pips4, pips5);
maxpool1.compute_h(pips5, pips6);
maxpool1.compute_v(pips6, pips7);
matmul0.compute_muladd(pips7, outs);
write_result<256, 10, 16>(out, outs);
```




The `MaxPool2x2` template class is a good example for task-level parallelism:

```cpp
template <typename T, int H, int W, int C>
class MaxPool2x2 {
private:
	T maxpool(const T val1, const T val2) {
		T oval;
		for (int z = 0; z < C; z++) {
#pragma HLS unroll
			uint2_t v1 = bit::getu<C>(val1, z);
			uint2_t v2 = bit::getu<C>(val2, z);
			bit::set<C>(oval, z, v1 > v2 ? v1 : v2);
		}
		return oval;
	}
public:
	void compute_h(fifo<T>& ins, fifo<T>& outs) {
		for (int xy = 0; xy < H * W; xy += 2) {
#pragma HLS pipeline
			T val1 = ins.read();
			T val2 = ins.read();
			T oval = maxpool(val1, val2);
			outs.write(oval);
		}
	}

	void compute_v(fifo<T>& ins, fifo<T>& outs) {
		T buf[W / 2];
#pragma HLS array_partition variable=buf

		for (int y = 0; y < H / 2; y++) {
#pragma HLS pipeline
			for (int x = 0; x < W / 2; x++) {
				T val = ins.read();
				buf[x] = val;
			}
			for (int x = 0; x < W / 2; x++) {
				T val1 = buf[x];
				T val2 = ins.read();
				T oval = maxpool(val1, val2);
				outs.write(oval);
			}
		}
	}
};
```

The `MaxPool2x2` class shows several key points of task-level parallelism, including:

- Task Decomposition: The 2D max pooling is split into horizontal and vertical operations (`compute_h` and `compute_v`), each task performs a specific part of the computation, and can be execute in parallel. The `compute_h` has no imtermediate storage, while `compute_v` use a buffer array to store the result of `compute_h`.

- Data Flow: Tasks use FIFOs to ensure efficient data transfer, which allow overlapped execution. When `compute_h` produces data to the FIFO, `compute_v` can start executing by reading data from the same FIFO. This allows concurrent execution of the two tasks without waiting for the entire `compute_h` operation to complete.

```cpp
	// call of maxpool method
#pragma HLS dataflow
	read_input<28, 28, 5, 5>(in, ins);
	conv0.compute<int2x16_t>(ins, pips1);
	maxpool0.compute_h(pips1, pips2);
	maxpool0.compute_v(pips2, pips3);
	// ... existing code ...
```

The Timeline Trace view in the Vitis Unified IDE C/RTL Cosimulation reports visually demonstrates the effects of task-level parallelism. 
![timeline trace](image/timeline.png)



## 2.4 Data-Level Parallelism

### Arbitrary Precision Type

Vitis HLS provides a C++ template class, `ap_[u]int<>`, that implements arbitrary precision (or bit-accurate) integer data types with consistent, bit-accurate behavior between software and hardware modeling. You can see 

In this Vitis HLS LeNet-5 implementation, we defined `int2_t` and `uint2_t` data types using `ap_int<2>` and `ap_uint<2>` to store int2 quantized data. Int2 quantization is an extreme form of quantization that aims to maximize data parallelism and minimize resource usage. `int2x25_t` and `int2x16_t` are also defined for data transfer.

```cpp
// lenet5.hpp

using int2_t = ap_int<2>;
using uint2_t = ap_uint<2>;
using int2x25_t = ap_uint<2 * KERNEL * KERNEL>;
using int2x16_t = ap_uint<2 * CHANNEL>;
template <class T>
using fifo = hls::stream<T>;

typedef ap_axis<32,0,0,0> int32_pkt;
```

This HLS LeNet-5 implementation use AXI-Stream to transfer data and the data bus width is 32-bit. Therefore, we can pack 16 int2 values into one 32-bit data, allowing transfer of 16 values in a single bus transaction, which greatly improves parallelism.   



### Pragmas for Data-Level Parallelism

In previous tutorials, we have covered Vitis HLS pragmas for data-level parallelism such as `array_partition`, `unroll`, and `pipeline`. You can view all Vitis HLS pragmas and their options used in the current HLS program in the Vitis Unified IDE under `C SYNTHESIS/REPORTS/Synthesis/Pragma Report`.




![pragma](image/pragma.png)

---------------------------------------
<p align="center">Copyright&copy; 2025 Advanced Micro Devices</p>